# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error
)

import lightgbm as lgb

### Define the data folder

In [15]:
DATA_DIR = Path(
    r"C:\Users\anupa\Downloads\Spotter"
)

OUTPUT_DIR = DATA_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)

Data folder: C:\Users\anupa\Downloads\Spotter
Output folder: C:\Users\anupa\Downloads\Spotter\outputs


### Load the three datasets and convert dates

In [17]:
tr = pd.read_csv(DATA_DIR / "train-test.csv")
va = pd.read_csv(DATA_DIR / "validation.csv")
ch = pd.read_csv(DATA_DIR / "december-chart-inputs.csv")

for d in (tr, va, ch):
    d["date"] = pd.to_datetime(d["date"])

print("Training data:", tr.shape)
print("Validation data:", va.shape)
print("December chart inputs:", ch.shape)

display(tr.head())

Training data: (48000, 14)
Validation data: (12000, 13)
December chart inputs: (31, 7)


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


### Create a city-to-coordinate lookup

In [18]:
city = pd.concat([
    tr[["pickup", "pickup_lat", "pickup_lon"]]
        .set_axis(["c", "lat", "lon"], axis=1),

    tr[["delivery", "delivery_lat", "delivery_lon"]]
        .set_axis(["c", "lat", "lon"], axis=1)
]).drop_duplicates("c").set_index("c")

display(city.head())

,lat,lon
c,,
Richmond,38.09122,-76.78906
Philadelphia,39.22317,-72.96710
Hartford,39.55328,-72.18051
Dallas,31.83025,-94.38343
Phoenix,28.75668,-115.87725


### Add missing coordinates to the December data

In [19]:
for p, pre in (
    ("pickup", "pickup"),
    ("delivery", "delivery")
):
    ch[pre + "_lat"] = ch[p].map(city["lat"])
    ch[pre + "_lon"] = ch[p].map(city["lon"])

print(
    ch[[
        "pickup", "pickup_lat", "pickup_lon",
        "delivery", "delivery_lat", "delivery_lon"
    ]].head()
)

      pickup  pickup_lat  pickup_lon    delivery  delivery_lat  delivery_lon
0  Lexington    36.99152   -84.99876  Fort Wayne      41.31561     -85.36206
1  Lexington    36.99152   -84.99876  Fort Wayne      41.31561     -85.36206
2  Lexington    36.99152   -84.99876  Fort Wayne      41.31561     -85.36206
3  Lexington    36.99152   -84.99876  Fort Wayne      41.31561     -85.36206
4  Lexington    36.99152   -84.99876  Fort Wayne      41.31561     -85.36206


### Prepare the market index and quote signal

In [20]:
mi = (
    pd.concat([
        tr[["date", "market_index"]],
        va[["date", "market_index"]]
    ])
    .groupby("date")["market_index"]
    .mean()
)

qs = pd.concat([
    tr[["date", "quote_signal"]],
    va[["date", "quote_signal"]]
])

qs_month_mean = (
    qs.groupby(qs["date"].dt.month)["quote_signal"]
    .mean()
)

ch["market_index"] = ch["date"].map(mi)
ch["quote_signal"] = (
    ch["date"].dt.month.map(qs_month_mean)
)

print("Market index values:", mi.shape[0])
print("Monthly quote-signal averages:")
display(qs_month_mean)

Market index values: 365
Monthly quote-signal averages:


date
1     2.068436
2     2.097130
3     2.179539
4     1.963388
5     1.912829
6     2.296007
7     1.922997
8     2.052200
9     2.199785
10    1.942758
11    2.055298
12    2.047581
Name: quote_signal, dtype: float64

### Calculate geographic distance

In [21]:
R = 3958.8

def hv(a, b, c, d):
    a, b, c, d = map(
        np.radians,
        [a, b, c, d]
    )

    x = (
        np.sin((c - a) / 2) ** 2
        + np.cos(a) * np.cos(c)
        * np.sin((d - b) / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(x))

### Create the feature-engineering function

In [24]:
def feat(d):
    X = pd.DataFrame(index=d.index)

    # 1. Geographic distance
    h = hv(
        d["pickup_lat"],
        d["pickup_lon"],
        d["delivery_lat"],
        d["delivery_lon"]
    )

    # 2. Compare reported distance with geographic distance
    ratio = d["distance"] / h

    bad = (
        ((ratio > 1.5) | (ratio < 1.0))
        & (d["distance"] > 70)
    ).values

    dist = np.where(
        bad,
        1.18 * h,
        d["distance"]
    )

    X["dist"] = dist
    X["logdist"] = np.log(dist)
    X["hav"] = h
    X["dist_bad"] = bad.astype(int)

    # 3. Shipment weight
    w = d["weight"].abs()

    X["w_missing"] = w.isna().astype(int)
    X["weight"] = w.fillna(med_w)
    X["w_neg"] = (d["weight"] < 0).astype(int)

    # 4. Market indicators
    X["mi_missing"] = d["market_index"].isna().astype(int)

    X["market_index"] = (
        d["market_index"]
        .fillna(d["date"].map(mi))
        .fillna(mi.mean())
    )

    X["quote_signal"] = d["quote_signal"]

    # 5. Calendar features
    X["month"] = d["date"].dt.month
    X["dow"] = d["date"].dt.dayofweek
    X["doy"] = d["date"].dt.dayofyear

    # 6. Pickup and delivery features
    for k in ("pickup", "delivery"):
        X[k + "_lat"] = d[k + "_lat"]
        X[k + "_lon"] = d[k + "_lon"]
        X[k] = d[k]

    # 7. Equipment type
    X["equipment"] = d["equipment"]

    # 8. Direction of the shipment
    X["dlat"] = d["delivery_lat"] - d["pickup_lat"]
    X["dlon"] = d["delivery_lon"] - d["pickup_lon"]

    return X

In [26]:
med_w = tr["weight"].abs().median()

print("Median absolute weight:", med_w)

Median absolute weight: 31496.0


### Generate features for all datasets

In [27]:
Xtr = feat(tr)
Xva = feat(va)
Xch = feat(ch)

print("Training features:", Xtr.shape)
print("Validation features:", Xva.shape)
print("December features:", Xch.shape)

display(Xtr.head())

Training features: (48000, 22)
Validation features: (12000, 22)
December features: (31, 22)


,dist,logdist,hav,dist_bad,w_missing,weight,w_neg,mi_missing,market_index,quote_signal,...,doy,pickup_lat,pickup_lon,pickup,delivery_lat,delivery_lon,delivery,equipment,dlat,dlon
0,274.3,5.614222,219.808768,0,0,30658.0,0,0,0.95684,2.39595,...,1,38.09122,-76.78906,Richmond,38.16908,-72.74564,Baltimore,Dry Van,0.07786,4.04342
1,280.5,5.636574,220.525617,0,0,17555.0,0,0,0.97623,2.43355,...,1,38.09122,-76.78906,Richmond,39.22317,-72.96710,Philadelphia,Reefer,1.13195,3.82196
2,967.8,6.875025,826.919847,0,0,31721.0,0,0,1.00971,1.84491,...,1,39.22317,-72.96710,Philadelphia,44.30296,-87.52871,Green Bay,Dry Van,5.07979,-14.56161
3,965.4,6.872543,840.540749,0,0,32333.0,0,0,0.94518,1.87712,...,1,39.55328,-72.18051,Hartford,34.84933,-86.28940,Atlanta,Dry Van,-4.70395,-14.10889
4,541.9,6.295081,434.163187,0,0,35183.0,0,0,0.98480,2.56300,...,1,31.83025,-94.38343,Dallas,35.29479,-88.08915,Nashville,Reefer,3.46454,6.29428


### Define the target and prepare categorical variables

In [41]:
rpm = tr["posted_rate"] / tr["distance"]

clean = ((rpm < 5) & (rpm > 0.9)).values

y = np.log(tr["posted_rate"])

print("Total training rows:", len(tr))
print("Clean training rows:", clean.sum())
print("Excluded from clean subset:", (~clean).sum())

Total training rows: 48000
Clean training rows: 47376
Excluded from clean subset: 624


In [42]:
cats = ["pickup", "delivery", "equipment"]

for X in (Xtr, Xva, Xch):
    for c in cats:
        X[c] = pd.Categorical(
            X[c],
            categories=sorted(
                set(tr[c].dropna().unique())
            )
        )

num = [
    c for c in Xtr.columns
    if c not in cats
]

print("Categorical features:", cats)
print("Number of numerical features:", len(num))
print("Numerical features:", num)

Categorical features: ['pickup', 'delivery', 'equipment']
Number of numerical features: 19
Numerical features: ['dist', 'logdist', 'hav', 'dist_bad', 'w_missing', 'weight', 'w_neg', 'mi_missing', 'market_index', 'quote_signal', 'month', 'dow', 'doy', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'dlat', 'dlon']


### Define the three machine-learning models

#### Ridge Regression

In [47]:
def ridge():
    pre = ColumnTransformer([
        (
            "n",
            StandardScaler(),
            [
                "logdist",
                "dist",
                "weight",
                "w_missing",
                "market_index",
                "quote_signal",
                "month",
                "dow",
                "dist_bad"
            ]
        ),
        (
            "c",
            OneHotEncoder(handle_unknown="ignore"),
            cats
        )
    ])

    return make_pipeline(
        pre,
        Ridge(alpha=3)
    )

### Random Forest Regression

In [48]:
def rf():
    pre = ColumnTransformer([
        ("n", "passthrough", num),
        (
            "c",
            OneHotEncoder(handle_unknown="ignore"),
            cats
        )
    ])

    return make_pipeline(
        pre,
        RandomForestRegressor(
            n_estimators=200,
            min_samples_leaf=5,
            max_features=0.5,
            n_jobs=-1,
            random_state=0
        )
    )

### LightGBM Regression

In [49]:
def gbm():
    return lgb.LGBMRegressor(
        objective="l1",
        n_estimators=800,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=30,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        verbose=-1,
        random_state=0
    )

### Define the time-based holdout

In [50]:
cut = pd.Timestamp("2025-09-01")

itr = (tr["date"] < cut).values
iho = (~itr)

print("Training-period rows:", itr.sum())
print("Holdout-period rows:", iho.sum())

print(
    "Holdout date range:",
    tr.loc[iho, "date"].min(),
    "to",
    tr.loc[iho, "date"].max()
)

Training-period rows: 38477
Holdout-period rows: 9523
Holdout date range: 2025-09-01 00:00:00 to 2025-10-31 00:00:00


### Create the model evaluation function

In [51]:
def ev(name, m):
    fit = itr & clean

    # Train the model
    m.fit(Xtr.loc[fit], y.loc[fit])

    # Predict on the holdout period
    p = np.exp(m.predict(Xtr.loc[iho]))

    # Actual posted rates
    t = tr.loc[iho, "posted_rate"].values

    # Identify clean holdout records
    c = clean[iho]

    if len(t) == 0:
        raise ValueError("The holdout period contains no rows.")

    if c.sum() == 0:
        raise ValueError("No clean rows are available in the holdout.")

    # Calculate evaluation metrics
    r = {
        "model": name,

        "MAE_all": mean_absolute_error(t, p),

        "RMSE_all": mean_squared_error(t, p) ** 0.5,

        "MedAE_all": median_absolute_error(t, p),

        "MAE_clean": mean_absolute_error(t[c], p[c]),

        "RMSE_clean": mean_squared_error(t[c], p[c]) ** 0.5,

        "MAPE_clean": (
            np.mean(
                np.abs(t[c] - p[c]) / t[c]
            ) * 100
        )
    }

    return r

### Train all three models and compare them

In [52]:
res = [
    ev("Ridge (log)", ridge()),
    ev("RandomForest", rf()),
    ev("LightGBM", gbm())
]

R_ = pd.DataFrame(res).round(2)

display(R_)

,model,MAE_all,RMSE_all,MedAE_all,MAE_clean,RMSE_clean,MAPE_clean
0,Ridge (log),105.29,633.06,32.98,53.21,159.49,2.31
1,RandomForest,112.47,635.95,36.55,60.39,164.58,2.51
2,LightGBM,109.04,634.75,35.09,57.05,163.09,2.38


### Create the time-based training and holdout split

In [53]:
cut = pd.Timestamp("2025-09-01")

itr = (tr["date"] < cut).values
iho = ~itr

fit = itr & clean

print("Training-period rows:", itr.sum())
print("Holdout-period rows:", iho.sum())
print("Clean training rows:", fit.sum())

Training-period rows: 38477
Holdout-period rows: 9523
Clean training rows: 37983


### Define the evaluation metrics

In [54]:
def metrics(p):
    t = tr.loc[iho, "posted_rate"].values
    c = clean[iho]

    return {
        "MAE_all": mean_absolute_error(t, p),

        "RMSE_all": mean_squared_error(t, p) ** 0.5,

        "MAE_clean": mean_absolute_error(t[c], p[c]),

        "RMSE_clean": mean_squared_error(t[c], p[c]) ** 0.5,

        "MAPE_clean": (
            np.mean(
                np.abs(t[c] - p[c]) / t[c]
            ) * 100
        )
    }

### Build the Hybrid model

In [55]:
class Hybrid:

    def fit(self, X, y):
        # Step 1: Train Ridge Regression
        self.r = ridge().fit(X, y)

        # Step 2: Calculate Ridge residuals
        res = y - self.r.predict(X)

        # Step 3: Train LightGBM on residuals
        self.g = lgb.LGBMRegressor(
            objective="l1",
            n_estimators=300,
            learning_rate=0.02,
            num_leaves=15,
            min_child_samples=100,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            reg_lambda=10,
            verbose=-1,
            random_state=0
        ).fit(X, res)

        return self

    def predict(self, X):
        # Add Ridge prediction and LightGBM correction
        return self.r.predict(X) + self.g.predict(X)

### Train and compare Ridge versus Hybrid

In [56]:
results = []

for name, model in [
    ("Ridge", ridge()),
    ("Hybrid", Hybrid())
]:
    # Train on clean records before the cutoff
    model.fit(Xtr.loc[fit], y.loc[fit])

    # Predict log rates, then convert to original scale
    predictions = np.exp(
        model.predict(Xtr.loc[iho])
    )

    # Evaluate performance
    scores = metrics(predictions)

    results.append({
        "model": name,
        **{k: round(v, 2) for k, v in scores.items()}
    })

comparison = pd.DataFrame(results)

display(comparison)

,model,MAE_all,RMSE_all,MAE_clean,RMSE_clean,MAPE_clean
0,Ridge,105.29,633.06,53.21,159.49,2.31
1,Hybrid,104.64,633.17,52.57,159.39,2.26


### Perform a Ridge alpha sweep

In [57]:
alpha_results = []

for alpha in (0.3, 1, 3, 10, 30):
    model = ridge()

    # Change the alpha parameter in the Ridge step
    model.steps[-1][1].set_params(alpha=alpha)

    # Train the model
    model.fit(Xtr.loc[fit], y.loc[fit])

    # Predict on the holdout
    predictions = np.exp(
        model.predict(Xtr.loc[iho])
    )

    # Calculate clean-set MAE
    score = metrics(predictions)["MAE_clean"]

    alpha_results.append({
        "alpha": alpha,
        "MAE_clean": round(score, 2)
    })

alpha_comparison = pd.DataFrame(alpha_results)

display(
    alpha_comparison.sort_values("MAE_clean")
)

,alpha,MAE_clean
0,0.3,53.20
1,1.0,53.20
2,3.0,53.21
3,10.0,53.25
4,30.0,53.41


### Save both comparison tables

In [58]:
comparison.to_csv(
    OUTPUT_DIR / "ridge_vs_hybrid.csv",
    index=False
)

alpha_comparison.to_csv(
    OUTPUT_DIR / "ridge_alpha_comparison.csv",
    index=False
)

print("Saved Ridge vs Hybrid comparison.")
print("Saved Ridge alpha comparison.")
print("Output folder:", OUTPUT_DIR)

Saved Ridge vs Hybrid comparison.
Saved Ridge alpha comparison.
Output folder: C:\Users\anupa\Downloads\Spotter\outputs


### Train the Hybrid model

In [59]:
m = Hybrid().fit(Xtr[clean], y[clean])

### Generate predictions

In [60]:
pv = np.exp(m.predict(Xva))
pc = np.exp(m.predict(Xch))

### Locate the input CSV files

In [61]:
DATA_DIR = Path.cwd()

print("Notebook working folder:", DATA_DIR)
print("Template file exists:",
      (DATA_DIR / "validation-predictions-template.csv").exists())
print("December input exists:",
      (DATA_DIR / "december-chart-inputs.csv").exists())

Notebook working folder: C:\Users\anupa\Downloads\Spotter
Template file exists: True
December input exists: True


### Load the validation template and verify row alignment

In [62]:
tmpl = pd.read_csv(DATA_DIR / "validation-predictions-template.csv")

assert (tmpl.load_id.values == va.load_id.values).all()

### Add predictions and save the validation file

In [63]:
OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tmpl["predicted_rate"] = pv.round(2)

tmpl.to_csv(
    OUTPUT_DIR / "validation_predictions.csv",
    index=False
)

### Check validation predictions

In [64]:
print(
    tmpl.shape,
    tmpl.predicted_rate.isna().sum(),
    tmpl.predicted_rate.describe().round(1).to_dict()
)

(12000, 2) 0 {'count': 12000.0, 'mean': 2391.6, 'std': 1381.5, 'min': 191.5, '25%': 1288.2, '50%': 2068.4, '75%': 3398.8, 'max': 6911.7}


### Examine predicted rate per unit of distance

In [65]:
print((pv / va.distance).describe().round(2).to_dict())

{'count': 12000.0, 'mean': 2.22, 'std': 0.26, 'min': 1.72, '25%': 2.03, '50%': 2.18, '75%': 2.36, 'max': 3.6}


### Load the December chart data and save predictions

In [66]:
out = pd.read_csv(DATA_DIR / "december-chart-inputs.csv")

out["predicted_rate"] = pc.round(2)

out.to_csv(
    OUTPUT_DIR / "december_chart_inputs.csv",
    index=False
)

### Inspect a sample of December predictions

In [67]:
print(
    out[["date", "predicted_rate"]]
    .iloc[::5]
    .to_string()
)

          date  predicted_rate
0   2025-12-01          837.08
5   2025-12-06          841.08
10  2025-12-11          855.40
15  2025-12-16          844.80
20  2025-12-21          837.75
25  2025-12-26          852.23
30  2025-12-31          855.32


### Check December row count and missing predictions

In [68]:
print(len(out), out.predicted_rate.isna().sum())

31 0


### Find the six lowest predicted rate-per-distance shipments

In [69]:
va["pred"] = pv
va["rpm"] = pv / va.distance

print(
    va.sort_values("rpm")
      .head(6)[
          ["pickup", "delivery", "distance", "weight",
           "equipment", "pred", "rpm"]
      ]
)

           pickup     delivery  distance   weight equipment         pred  \
10937    Syracuse  Los Angeles    2841.1   7696.0   Dry Van  4890.911698   
8044         Reno       Albany    2894.0  21494.0   Dry Van  4988.975962   
7286     Syracuse         Reno    2571.2  24893.0   Dry Van  4435.361312   
7363   Harrisburg      Phoenix    2639.0  12343.0   Dry Van  4552.400195   
1068         Reno       Albany    2870.2  18364.0   Dry Van  4952.276566   
1683   Harrisburg         Reno    2513.6  21991.0   Dry Van  4338.473413   

            rpm  
10937  1.721485  
8044   1.723903  
7286   1.725016  
7363   1.725047  
1068   1.725412  
1683   1.726000  


### Investigate those shipments' distance features

In [70]:
low_rpm = va.sort_values("rpm").head(6)

print(
    Xva.loc[
        low_rpm.index,
        ["dist", "hav", "dist_bad"]
    ]
)

         dist          hav  dist_bad
10937  2841.1  2412.826316         0
8044   2894.0  2437.582911         0
7286   2571.2  2160.518852         0
7363   2639.0  2266.083324         0
1068   2870.2  2437.582911         0
1683   2513.6  2137.612462         0


### Count flagged distances

In [71]:
print("Validation rows with rpm < 1:", (va.rpm < 1).sum())
print("Validation rows with dist_bad = 1:", (Xva.dist_bad == 1).sum())
print("Training rows with dist_bad = 1:", (Xtr.dist_bad == 1).sum())

Validation rows with rpm < 1: 0
Validation rows with dist_bad = 1: 39
Training rows with dist_bad = 1: 187
